<a href="https://colab.research.google.com/github/PALAK7890/SectionA_Team14_JobMarketTrends/blob/main/02_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📂 /kaggle/input/linkedin-job-postings
   📄 postings.csv
📂 /kaggle/input/linkedin-job-postings/companies
   📄 companies.csv
   📄 company_industries.csv
   📄 company_specialities.csv
   📄 employee_counts.csv
📂 /kaggle/input/linkedin-job-postings/mappings
   📄 industries.csv
   📄 skills.csv
📂 /kaggle/input/linkedin-job-postings/jobs
   📄 benefits.csv
   📄 job_skills.csv
   📄 job_industries.csv
   📄 salaries.csv


In [ ]:
import kagglehub
import pandas as pd
import os

# Download dataset
path = kagglehub.dataset_download("arshkon/linkedin-job-postings")
print("Path to dataset files:", path)

data_path = path

# Load datasets
jobs = pd.read_csv(os.path.join(data_path, 'postings.csv'))
companies = pd.read_csv(os.path.join(data_path, 'companies', 'companies.csv'))
employees = pd.read_csv(os.path.join(data_path, 'companies', 'employee_counts.csv'))
benefits = pd.read_csv(os.path.join(data_path, 'jobs', 'benefits.csv'))
salaries = pd.read_csv(os.path.join(data_path, 'jobs', 'salaries.csv'))
job_skills = pd.read_csv(os.path.join(data_path, 'jobs', 'job_skills.csv'))

print("Jobs:", jobs.shape)
print("Companies:", companies.shape)
print("Employees:", employees.shape)
print("Benefits:", benefits.shape)
print("Salaries:", salaries.shape)
print("Job Skills:", job_skills.shape)

Using Colab cache for faster access to the 'linkedin-job-postings' dataset.
Path to dataset files: /kaggle/input/linkedin-job-postings
Jobs: (123849, 31)
Companies: (24473, 10)
Employees: (35787, 4)
Benefits: (67943, 3)
Salaries: (40785, 8)
Job Skills: (213768, 2)


In [ ]:
import os

for root, dirs, files in os.walk(data_path):
    print("📂", root)
    for file in files:
        print("   📄", file)

📂 /kaggle/input/linkedin-job-postings
   📄 postings.csv
📂 /kaggle/input/linkedin-job-postings/companies
   📄 companies.csv
   📄 company_industries.csv
   📄 company_specialities.csv
   📄 employee_counts.csv
📂 /kaggle/input/linkedin-job-postings/mappings
   📄 industries.csv
   📄 skills.csv
📂 /kaggle/input/linkedin-job-postings/jobs
   📄 benefits.csv
   📄 job_skills.csv
   📄 job_industries.csv
   📄 salaries.csv


In [ ]:
jobs.head()
jobs.info()
jobs.isnull().sum().sort_values(ascending=False).head(20)
jobs.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

np.int64(0)

In [ ]:
import numpy as np

# Create a copy of the jobs dataframe so the original data remains unchanged
df = jobs.copy()

In [ ]:
df = df.drop_duplicates()

# NUMERIC COLUMN CLEANING

In [ ]:

# List of columns that should be numeric
numeric_cols = [
    'job_id', 'company_id', 'views', 'med_salary', 'min_salary', 'max_salary',
    'applies', 'original_listed_time', 'remote_allowed', 'expiry',
    'closed_time', 'listed_time', 'normalized_salary', 'zip_code', 'fips'
]

# Convert numeric columns to proper numeric format
# errors='coerce' converts invalid values into NaN
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

#  DATE/TIME CONVERSION


In [ ]:
# Convert ID and location code columns to nullable integer type
# Int64 allows missing values, unlike normal int
df['company_id'] = df['company_id'].astype('Int64')

df['zip_code'] = df['zip_code'].astype('Int64')
df['fips'] = df['fips'].astype('Int64')

In [ ]:
# LinkedIn dataset stores some time columns in milliseconds
time_cols = ['original_listed_time', 'expiry', 'closed_time', 'listed_time']

# Convert millisecond timestamp columns into readable datetime format
for col in time_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', unit='ms')

# TEXT COLUMN CLEANING

In [1]:
# List of text-based columns

text_cols = [
    'company_name', 'title', 'description', 'pay_period', 'location',
    'formatted_work_type', 'job_posting_url', 'application_url',
    'application_type', 'formatted_experience_level', 'skills_desc',
    'posting_domain', 'work_type', 'currency', 'compensation_type'
]
# Convert text columns to string format and remove extra spaces

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype('string').str.strip()

# Replace empty or invalid text values with proper missing value marker

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].replace(['', 'nan', 'None'], pd.NA)


NameError: name 'df' is not defined

#  HANDLE IMPORTANT MISSING VALUES

In [ ]:
df['remote_allowed'] = df['remote_allowed'].fillna(0)
df['remote_allowed'] = df['remote_allowed'].astype(int)

In [ ]:
df['sponsored'] = df['sponsored'].fillna(0).astype(int)

In [ ]:
df['company_name'] = df['company_name'].fillna('Unknown')
df['formatted_experience_level'] = df['formatted_experience_level'].fillna('Not Specified')
df['posting_domain'] = df['posting_domain'].fillna('Unknown')
df['skills_desc'] = df['skills_desc'].fillna('Not Provided')

FEATURE ENGINEERING

In [ ]:
df['avg_salary'] = df[['min_salary', 'max_salary']].mean(axis=1)

df['is_remote'] = df['remote_allowed'].map({1: 'Remote Allowed', 0: 'Onsite/Not Specified'})


In [ ]:
df['views'] = df['views'].fillna(0)
df['applies'] = df['applies'].fillna(0)
df['demand_score'] = df['views'] + df['applies']

In [ ]:

df['year'] = df['listed_time'].dt.year
df['month'] = df['listed_time'].dt.month
df['month_name'] = df['listed_time'].dt.month_name()

In [ ]:
df['job_active_days'] = (df['expiry'] - df['listed_time']).dt.days

In [ ]:
def simplify_experience(x):
    if pd.isna(x):
        return 'Unknown'
    x = str(x).lower()
    if 'intern' in x:
        return 'Internship'
    elif 'entry' in x:
        return 'Entry'
    elif 'associate' in x:
        return 'Associate'
    elif 'mid' in x:
        return 'Mid'
    elif 'senior' in x:
        return 'Senior'
    elif 'director' in x:
        return 'Director'
    elif 'executive' in x:
        return 'Executive'
    else:
        return 'Other'

df['experience_group'] = df['formatted_experience_level'].apply(simplify_experience)

In [ ]:
print(df.shape)
print(df.info())
print(df.isnull().sum().sort_values(ascending=False).head(15))

(123849, 39)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 39 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   job_id                      123849 non-null  int64         
 1   company_name                123849 non-null  string        
 2   title                       123849 non-null  string        
 3   description                 123842 non-null  string        
 4   max_salary                  29793 non-null   float64       
 5   pay_period                  36073 non-null   string        
 6   location                    123849 non-null  string        
 7   company_id                  122132 non-null  Int64         
 8   views                       123849 non-null  float64       
 9   med_salary                  6280 non-null    float64       
 10  min_salary                  29793 non-null   float64       
 11  formatted_work_type       

In [ ]:
df[['listed_time', 'expiry', 'job_active_days']].describe(include='all')
print(df['job_active_days'].min(), df['job_active_days'].max())
print((df['job_active_days'] < 0).sum())



3 180
0


In [ ]:
skills_df = df[['job_id', 'title']].merge(job_skills, on='job_id', how='left')
skills_df['skill_abr'].value_counts().head(20)

,count
skill_abr,
IT,25256
SALE,21193
MGMT,20385
MNFC,17728
HCPR,16675
BD,13304
ENG,12530
OTHR,12314
FIN,8011


In [ ]:
df['normalized_salary'] = df.groupby(
    ['experience_group', 'formatted_work_type', 'location']
)['normalized_salary'].transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
dashboard_df = df[[
    'job_id',
    'company_name',
    'title',
    'location',
    'formatted_work_type',
    'experience_group',
    'is_remote',
    'views',
    'applies',
    'demand_score',
    'sponsored',
    'normalized_salary',
    'salary_available',
    'pay_period',
    'year',
    'month',
    'month_name',
    'job_active_days'
]].copy()
dashboard_df['pay_period'] = dashboard_df['pay_period'].fillna('Not Specified')
dashboard_df['normalized_salary'] = pd.to_numeric(dashboard_df['normalized_salary'], errors='coerce')
dashboard_df.to_csv("linkedin_jobs_dashboard_ready.csv", index=False)
print("Saved: linkedin_jobs_dashboard_ready.csv")

Saved: linkedin_jobs_dashboard_ready.csv


In [ ]:
dashboard_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   job_id               123849 non-null  int64  
 1   company_name         123849 non-null  string 
 2   title                123849 non-null  string 
 3   location             123849 non-null  string 
 4   formatted_work_type  123849 non-null  string 
 5   experience_group     123849 non-null  object 
 6   is_remote            123849 non-null  object 
 7   views                123849 non-null  float64
 8   applies              123849 non-null  float64
 9   demand_score         123849 non-null  float64
 10  sponsored            123849 non-null  int64  
 11  normalized_salary    96279 non-null   float64
 12  salary_available     123849 non-null  bool   
 13  pay_period           123849 non-null  string 
 14  year                 123849 non-null  int32  
 15  month            